Lemma 2.5 [22, 3, 13](CSS construction) 

Let $C_1$ and $C_2$ denote two classical linear codes with parameters $[n, k_1, d_1]_q$ and $[n, k_2, d_2]_q$, respectively, such that
$C_1 \subset C_2$. Then there exists an $[[n, k = k_2 − k_1, d]]_q$
quantum code, where $d =\text{min}\{wt(c)|c \in (C_2\setminus C_1) \cup (C^{\perp}_1 \setminus  C^{\perp}_2)\}$.


[3] @article{calderbank1998quantum,
  title={Quantum error correction via codes over GF (4)},
  author={Calderbank, A Robert and Rains, Eric M and Shor, Peter M and Sloane, Neil JA},
  journal={IEEE Transactions on Information Theory},
  volume={44},
  number={4},
  pages={1369--1387},
  year={1998},
  publisher={IEEE}
}

[13] @article{ketkar2006nonbinary,
  title={Nonbinary stabilizer codes over finite fields},
  author={Ketkar, Avanti and Klappenecker, Andreas and Kumar, Santosh and Sarvepalli, Pradeep Kiran},
  journal={IEEE transactions on information theory},
  volume={52},
  number={11},
  pages={4892--4914},
  year={2006},
  publisher={IEEE}
}

In [1]:
import random
import subprocess
def randomD(a,b,c,iavoid=0):
    #print("iavoid=",iavoid)
    lst = [i for i in range(a,b+1)]
    if iavoid!=None:
        lst = [i for i in lst if i !=1+iavoid]
    #print(lst)
    result = random.sample(lst, c)
    result.sort()
    return result
    
def getrandomG(np,maxd):
    G=[0]*np
    i = random.randint(0,np-1)
    j=random.randint(1,maxd)
    G[i]=j
    return G[0:i+1],i

getrandomG(6,5)

([0, 5], 1)

In [2]:
class Gene:
    def __init__(
        self,
        gene_text : str  
    ):
        self.gene_text=gene_text
        self.success=None
        self.distance=None
        self.NrRatpl=None
        self.PLACES=None
        self.Get_NrRatpl_PLACES()
        #print(self.gene_txt)
    def get_distance(self,i):
        #print("get_distance")
        self.distance=i
    def get_result(self,txt):
        self.result=txt
    def show_gene(self):
        print(self.gene_text)
    def Get_NrRatpl_PLACES(self):
        AS=self.gene_text.split("\n") 
        f=open("A_HC.txt","w")
        for u in AS:
            if u.find("def ER=")>=0:
                break
            print(u,file=f)
        print("print(\"---HC----\");HC[3];print(\"---HC----\");",file=f)
        f.close()
        result_HC = subprocess.run(["Singular -q A_HC.txt"], shell=True,capture_output=True, text=True)
        NrRatpl=None
        PLACES=[]
        HCFOUND=0
        for u in result_HC.stdout.split("\n"):
            #print(u)
            if u.find("NrRatPl")>=0:
                NrRatpl=u
            if u.find("---HC---")>=0:
                HCFOUND+=1
            if HCFOUND==1:
                PLACES.append(u)
        
        NrRatpl=int(NrRatpl.split()[-1])
        
        PLACES=[eval("("+u+")") for u in PLACES[1:] if u.find(":")<0]
        self.NrRatpl=NrRatpl
        self.PLACES=PLACES
        return NrRatpl,PLACES
        
    def RandomCSS(self,randomized=True):
        NrRatpl,PLACES=self.Get_NrRatpl_PLACES()
        AS=self.gene_text.split("\n")
        #print(AS)
        if randomized==True:
            OK=False
            while OK==False:
                AT=open("AT_.txt","w")
                iavoid=None
                for u in AS:
                    if u.find("intvec G")>=0 and randomized==True:
                        randomg,iavoid=getrandomG(1,4)
                        #print(randomg,iavoid)
                        w=str(randomg).replace("]","").replace("[","")
                        print("intvec G="+w+";",file=AT)   
                        #print("intvec G="+w+";",iavoid)   
                    elif u.find("intvec D")>=0 and randomized==True:
                        w=str(randomD(1,9,6,iavoid)).replace("]","").replace("[","")
                        #print(w)
                        print("intvec D="+w+";",file=AT)
                        #print("intvec D="+w+";")
                    else:
                        print(u,file=AT)
                AT.close()
            
                
                # コマンド実行
                result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
                
                #print(result.stdout)   # 標準出力
                if result.stdout.find("Vector basis successfully computed")>=0:
                    #print("OK")
                    OK=True
                    self.success=True
                else:
                    pass
                    OK=False
                    #print("FAILED")
                    f=open("AT_.txt","r")
                    #print(f.readline())
                    f.close()
                if result.stdout.find("wrong range")>=0:
                    break
                #print(result.stderr)   # エラー出力
                #print(result.returncode)  # 終了コード
        else:
            AT=open("AT_.txt","w")
            iavoid=None
            for u in AS:
                print(u,file=AT)
            AT.close()
            result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
            
            #print(result.stdout)   # 標準出力
            if result.stdout.find("Vector basis successfully computed")>=0:
                #print("OK")
                self.success=True
                OK=True
            else:
                pass
                #OK=False
                #print("FAILED")
                #f=open("AT_.txt","r")
                #print(f.readline())
                #f.close()
        if result.stdout.find("Vector basis successfully computed")>=0:
            #print("OK")
            self.success=True
            G.get_result(result.stdout)
            G.get_distance(int(result.stdout.split("\n")[-2]))
            OK=True
        else:
            pass
            #OK=False
            #G.get_result(result.stdout)
            #G.get_distance=None
            #print("FAILED")
            #f=open("AT_.txt","r")
            #print(f.readline())
            #f.close()



In [3]:
G=Gene(A0+C)
#print(G.gene_text)
#print(G.Get_NrRatpl_PLACES())
#G.RandomCSS()
#print(G.result.split("\n")[-2])
print(vars(G))

NameError: name 'A0' is not defined

In [4]:
import subprocess
result_HC = subprocess.run(["Singular -q A_HC.txt"], shell=True,capture_output=True, text=True)
NrRatpl=None
PLACES=[]
HCFOUND=0
for u in result_HC.stdout.split("\n"):
    #print(u)
    if u.find("NrRatPl")>=0:
        NrRatpl=u
    if u.find("---HC---")>=0:
        HCFOUND+=1
    if HCFOUND==1:
        PLACES.append(u)

NrRatpl=int(NrRatpl.split()[-1])

PLACES=[eval("("+u+")") for u in PLACES[1:] if u.find(":")<0]

NrRatpl,PLACES

(9, [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)])

In [5]:
G=[0]*len(PLACES)
G

[0, 0, 0, 0, 0, 0]

In [6]:
A_HC="LIB \"mybrnoeth.lib\";\
ring s=2,(x,y),lp;\
list HC=Adj_div(x3+y2+y);\
HC=NSplaces(1..2,HC);\
HC=extcurve(2,HC);\
print(\"---HC----\");\
HC[3];\
print(\"---HC----\");"


In [7]:
A0=\
"LIB \"mybrnoeth.lib\";\n\
LIB \"myprocs.txt\";\n\
ring s=2,(x,y),lp;\n\
list HC=Adj_div(x3+y2+y);\n\
HC=NSplaces(1..2,HC);\n\
HC=extcurve(2,HC);\n\
def ER=HC[1][4];\n\
setring ER;\n\
intvec G=5;\n\
intvec D=2,3,4,5,6,7,8,9; \n\
matrix C=AGcode_L(G,D,HC);\n\
print(C);\n\
print(\"Hamming_wt\");\n\
print(min_wt_rmat(C));\n\
"


In [8]:
C="print(\"C\");\
print(C);\
print(\"check_fully_one_rows\");\
intvec a=check_fully_one_rows(C);\
print(a);\
if (nrows(C)>1){C=MySubmat(a,C);\
print(\"C [The row (1,1,...,1) is removed]\");\
print(C);}\
print(\"Check_fully_zero_columns\");\
print(check_fully_zero_columns(C));\
intvec b=check_fully_zero_columns(C);\
if (1){C=MySubmat_cols(b, C);\
print(\"C [Fully zero colums are removed.]\");\
print(C);}\
print(my_min_wt_rmat(C));\
print(\"SC\");\
intvec m=1,2;\
matrix SC=MySubmat(m,C);\
print(SC);\
print(\"Ker(C)\");\
print(MyKer(C));\
print(\"Ker(SC)\");\
print(MyKer(SC));\
matrix KC=MyKer(C);\
matrix KSC=MyKer(SC);\
print(\"KSC\\KC\");\
print(MySupplement(KC,KSC));\
print(my_min_wt_rmat(MySupplement(KC,KSC)));\
print(\"C\\SC\");\
print(MySupplement(SC,C));\
print(my_min_wt_rmat(MySupplement(SC,C)));\
matrix CONCAT=concat(transpose(MySupplement(KC,KSC)),transpose(MySupplement(SC,C)));\
CONCAT=transpose(CONCAT);\
print(\"KSC\\KC + C\\SC\");\
print(CONCAT);\
print(\"rank(CONCAT)=\"+string(mat_rank(CONCAT)));\
print(my_min_wt_rmat(CONCAT));\
quit;"

In [9]:
G=Gene(A0.replace("\n","")+C)
GENETEXT=[u+";" for u in G.gene_text.split(";")]
for u in GENETEXT:
    if u.find("Adj_div")>=0:
        print(u)
    if u.find("intvec G")>=0:
        print(u)
    if u.find("intvec D")>=0:
        print(u)
        

AttributeError: 'NoneType' object has no attribute 'split'

In [10]:
G=Gene(A0+C)
G.Get_NrRatpl_PLACES(A_HC)

TypeError: Gene.Get_NrRatpl_PLACES() takes 1 positional argument but 2 were given

In [11]:
def disj_divs (H,P,auxIV,NrRatpl,PLACES):
# USGE:
#    For example,
#    H=[1, 0, 0, 0, 1] : G, INDEX TO PLACES (AS A LIST) 
#    P=[1, 6] : D, INDEX TO POINTS.
#    PLACES=[(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)] : List [...(deg., seq num.)..] (
#    POINTS ARE THE REARANGEMENT OF PLACES WHERE A PLACE (of degree d) YIELDS d points. 
#    viz., POINTS=[1|2|3|4,5|6,7|8,9]
#    THEREFORE H REFERS TO POINT 1, 6,7.
#   IT CHECKS WHETHER H AND H HAVE OVERLAPS ON SOME POINTS.
#   ALSO IT RETURNS THE POINTS CORRESPINDING TO THE GIVEN H, THE PLACES CORRESPINDING TO THE GIVEN H, THE PLACES CORRESPONDING TO THE GIVEN P.
#   BY COMPARING THOSE DATA, WE CAN SHAVE OFF THE OVERLAP BETWEEN H AND P. 

    if auxIV!=None:
        auxIVD=dict()
        for u in PLACES:
            auxIVD[u[0]]=0
        for u in PLACES:
            auxIVD[u[0]]+=1
        auxIV=[auxIVD[u] for u in auxIVD.keys()]
    s1=len(H);
    s2=len(P);
    s=2 
    #print("H (G)="+str(H));
    #print("P (POINTS)="+str(P));
    #print("PLACES="+str(PLACES));
    #print("s="+str(s));
    PLACES_POINT=dict()
    for u in PLACES:
        PLACES_POINT[u]=list()
    DIVISION=[u[0] for u in PLACES]
    #print(DIVISION)
    counter=1
    for u,v in zip(PLACES,DIVISION):
        for _ in range(v):
            PLACES_POINT[u].append(counter)
            counter+=1
    #print("PLACES->POINT",PLACES_POINT)    
    PLACES_POINT_KEYS=list(PLACES_POINT.keys())
    PLACES_POINT_KEYS.sort()
    POINTS_PLACE=dict()
    for u in PLACES_POINT.keys():
        for w in PLACES_POINT[u]:
            POINTS_PLACE[w]=u
    #print("POINTS->PLACE",POINTS_PLACE)

    H_CORRESPONDING_POINTS=list()
    H_CORRESPONDING_PLACES=list()
    for u in range(len(H)):
        if H[u]>0:
            ky=PLACES[u]
            H_CORRESPONDING_POINTS.extend(PLACES_POINT[ky])
            H_CORRESPONDING_PLACES.append(ky)
    #print("H->POINTS:",H_CORRESPONDING_POINTS)
    #print("H->PLACES:",H_CORRESPONDING_PLACES)
    
    P_CORRESPONDING_PLACES=list()
    for u in P:
        P_CORRESPONDING_PLACES.append(POINTS_PLACE[u])
    P_CORRESPONDING_PLACES=list(set(P_CORRESPONDING_PLACES))
    P_CORRESPONDING_PLACES.sort()
    #print("P->PLACES:",P_CORRESPONDING_PLACES)      
    return len(list( set(P) & set(H_CORRESPONDING_POINTS)))>0,H_CORRESPONDING_POINTS,H_CORRESPONDING_PLACES,P_CORRESPONDING_PLACES
        
J,HasPO,HasPL,PasPL=disj_divs ([1,1,1,1,1,1],[1,2,3,4,5,6,7,8,9],None,NrRatpl,PLACES)
J,HasPO,HasPL,PasPL
#disj_divs ([1,0,0,0,1],[1,6],None,NrRatpl,PLACES)

(True,
 [1, 2, 3, 4, 5, 6, 7, 8, 9],
 [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)],
 [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)])

In [12]:
P=[1,3,4,7,8]
J,HasPO,HasPL,PasPL=disj_divs ([1,1,1,1,0,1],P,None,NrRatpl,PLACES)
J,PasPL

(True, [(1, 1), (1, 3), (2, 1), (2, 2), (2, 3)])

In [13]:
def PallowedByH(HasPO):
    POpossible=[u+1 for u in range(NrRatpl)]
    return list(set(POpossible)-set(HasPO))

print(PallowedByH(HasPO))    

def HallowedByP(PLACES,PasPL):
    R=list(set(PLACES)-set(PasPL))
    R.sort()
    return R

print(HallowedByP(PLACES,PasPL))    


def GetRandomG(PLACES,NL2,GMAX=10,ALLOWED=PLACES):
    R=[0]*len(PLACES)
    #print(ALLOWED,PLACES)
    T=[i+1 for i in range(len(PLACES))]
    #print(T)
    for i in random.sample(ALLOWED,min(NL2,len(ALLOWED))):
        j=PLACES.index(i)
        R[j]=random.randint(1,GMAX)
    return R

ALLOWED=set(PLACES)-set(PasPL)
ALLOWED=list(ALLOWED)
print("ALLOWED=",ALLOWED)
#PLACES[ALLOWED]->random G
D=GetRandomG(PLACES,2,GMAX=3,ALLOWED=ALLOWED)
while sum(D) >4:
    D=GetRandomG(PLACES,2,GMAX=3,ALLOWED=ALLOWED)
print(D)


D=GetRandomG(PLACES,3,GMAX=3)
while sum(D) >4:
    D=GetRandomG(PLACES,3,GMAX=3)
print(D)

[6, 7]
[(1, 2)]
ALLOWED= [(1, 2)]
[0, 3, 0, 0, 0, 0]
[2, 1, 0, 0, 0, 1]


In [14]:
disj_divs ([4],[2,3,4,5,6,7],None,NrRatpl,PLACES)

(False, [1], [(1, 1)], [(1, 2), (1, 3), (2, 1), (2, 2)])

In [15]:
randomD(0,9,5)

[0, 2, 3, 5, 7]

In [16]:
LIB "mybrnoeth.lib";
ring s=13,(x,y),lp;
list HC=Adj_div(x3+y2+y);
HC=NSplaces(1..2,HC);
HC=extcurve(2,HC);


print("---HC----");
HC[3];
print("---HC----");

SyntaxError: invalid syntax (1061416419.py, line 1)

In [17]:
C.

SyntaxError: invalid syntax (818754905.py, line 1)

In [18]:
A=A0+C
A

'LIB "mybrnoeth.lib";\nLIB "myprocs.txt";\nring s=2,(x,y),lp;\nlist HC=Adj_div(x3+y2+y);\nHC=NSplaces(1..2,HC);\nHC=extcurve(2,HC);\ndef ER=HC[1][4];\nsetring ER;\nintvec G=5;\nintvec D=2,3,4,5,6,7,8,9; \nmatrix C=AGcode_L(G,D,HC);\nprint(C);\nprint("Hamming_wt");\nprint(min_wt_rmat(C));\nprint("C");print(C);print("check_fully_one_rows");intvec a=check_fully_one_rows(C);print(a);if (nrows(C)>1){C=MySubmat(a,C);print("C [The row (1,1,...,1) is removed]");print(C);}print("Check_fully_zero_columns");print(check_fully_zero_columns(C));intvec b=check_fully_zero_columns(C);if (1){C=MySubmat_cols(b, C);print("C [Fully zero colums are removed.]");print(C);}print(my_min_wt_rmat(C));print("SC");intvec m=1,2;matrix SC=MySubmat(m,C);print(SC);print("Ker(C)");print(MyKer(C));print("Ker(SC)");print(MyKer(SC));matrix KC=MyKer(C);matrix KSC=MyKer(SC);print("KSC\\KC");print(MySupplement(KC,KSC));print(my_min_wt_rmat(MySupplement(KC,KSC)));print("C\\SC");print(MySupplement(SC,C));print(my_min_wt_rmat(My

In [19]:
C=C.replace(";",";\n")

In [20]:
print(C)

print("C");
print(C);
print("check_fully_one_rows");
intvec a=check_fully_one_rows(C);
print(a);
if (nrows(C)>1){C=MySubmat(a,C);
print("C [The row (1,1,...,1) is removed]");
print(C);
}print("Check_fully_zero_columns");
print(check_fully_zero_columns(C));
intvec b=check_fully_zero_columns(C);
if (1){C=MySubmat_cols(b, C);
print("C [Fully zero colums are removed.]");
print(C);
}print(my_min_wt_rmat(C));
print("SC");
intvec m=1,2;
matrix SC=MySubmat(m,C);
print(SC);
print("Ker(C)");
print(MyKer(C));
print("Ker(SC)");
print(MyKer(SC));
matrix KC=MyKer(C);
matrix KSC=MyKer(SC);
print("KSC\KC");
print(MySupplement(KC,KSC));
print(my_min_wt_rmat(MySupplement(KC,KSC)));
print("C\SC");
print(MySupplement(SC,C));
print(my_min_wt_rmat(MySupplement(SC,C)));
matrix CONCAT=concat(transpose(MySupplement(KC,KSC)),transpose(MySupplement(SC,C)));
CONCAT=transpose(CONCAT);
print("KSC\KC + C\SC");
print(CONCAT);
print("rank(CONCAT)="+string(mat_rank(CONCAT)));
print(my_min_wt_rmat(CONCAT));
quit;



In [21]:
for u in C.split(";"):
    print(u+";")

print("C");

print(C);

print("check_fully_one_rows");

intvec a=check_fully_one_rows(C);

print(a);

if (nrows(C)>1){C=MySubmat(a,C);

print("C [The row (1,1,...,1) is removed]");

print(C);

}print("Check_fully_zero_columns");

print(check_fully_zero_columns(C));

intvec b=check_fully_zero_columns(C);

if (1){C=MySubmat_cols(b, C);

print("C [Fully zero colums are removed.]");

print(C);

}print(my_min_wt_rmat(C));

print("SC");

intvec m=1,2;

matrix SC=MySubmat(m,C);

print(SC);

print("Ker(C)");

print(MyKer(C));

print("Ker(SC)");

print(MyKer(SC));

matrix KC=MyKer(C);

matrix KSC=MyKer(SC);

print("KSC\KC");

print(MySupplement(KC,KSC));

print(my_min_wt_rmat(MySupplement(KC,KSC)));

print("C\SC");

print(MySupplement(SC,C));

print(my_min_wt_rmat(MySupplement(SC,C)));

matrix CONCAT=concat(transpose(MySupplement(KC,KSC)),transpose(MySupplement(SC,C)));

CONCAT=transpose(CONCAT);

print("KSC\KC + C\SC");

print(CONCAT);

print("rank(CONCAT)="+string(mat_rank(CONCAT)));

print(m

In [22]:
import subprocess
A=A0+C
A
AS=A.split("\n")
#print(AS)


current_max_distance=0
def RandomCSS_D_A():  
    global current_max_distance
    AT=open("AT_.txt","w")
    iavoid=None
    HasPO=None
    HasPL=None
    PasPL=None
    for u in AS:
    # WARNINGS: G  should satisfy @math{ 2*genus-2 < deg(G) < size(D) }, which is
    #           not checked by the algorithm.
        size_D=5
        if u.find("intvec G")>=0:
            D=GetRandomG(PLACES,2)
            while sum(D) >=size_D:
                D=GetRandomG(PLACES,2)
            randomg=D
            #print(randomg)
            J,HasPO,HasPL,PasPL=disj_divs (randomg,[l for l in range(1,NrRatpl+1)],None,NrRatpl,PLACES)
            #print(J,HasPO,HasPL,PasPL)
            w=str(randomg).replace("]","").replace("[","")
            print("intvec G="+w+";",file=AT)   
            #print("intvec G="+w+";")   
        elif u.find("intvec D")>=0:
            w=PallowedByH(HasPO)
            #print("w(allowed)=",w)
            wr=random.sample(w,size_D)
            wr.sort()
            wr
    #        w=str(randomD(1,9,6,iavoid)).replace("]","").replace("[","")
            w=str(wr).replace("]","").replace("[","")
            #print("w=",w)
            print("intvec D="+w+";",file=AT)
            #print("intvec D="+w+";")
        elif u.find("intvec m")>=0:
            intvecm=random.rample([i for i in range(1,4)],2)
            w=str(intvecm).replace("]","").replace("[","")
            print("intvec m="+w+";",file=AT)
        else:
            print(u,file=AT)
    AT.close()

    AT=open("AT_.txt","r")
    gene_text=AT.read()
    AT.close()
    G=Gene(gene_text)
    
    # コマンド実行
    result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
    if result.stdout.find("Vector basis successfully computed")>=0:
        print("OK")
        G.get_result(result.stdout)
        G.get_distance=int(result.stdout.split("\n")[-2])
        print("distance=",result.stdout.split("\n")[-2],"current_max_distance=",current_max_distance)   # 標準出力
        if int(result.stdout.split("\n")[-2])>current_max_distance:
            current_max_distance=int(result.stdout.split("\n")[-2])
        OK=True
    else:
        pass
        OK=False
        print("FAILED")
        f=open("AT_.txt","r")
        print(f.readline())
        f.close()
    
    #print(result.stderr)   # エラー出力
    #print(result.returncode)  # 終了コード


def Mutate(G0):  
    global current_max_distance
    AT=open("AT_.txt","w")
    iavoid=None
    HasPO=None
    HasPL=None
    PasPL=None
    #print(G0.gene_text)
    AS=G0.gene_text.split("\n")
    #AT=G0.gene_text
    #print(AS)
    for u in AS:
        #print(u)
    # WARNINGS: G  should satisfy @math{ 2*genus-2 < deg(G) < size(D) }, which is
    #           not checked by the algorithm.
        size_D=5
        if u.find("intvec G")>=0:
            D=GetRandomG(PLACES,2)
            while sum(D) >=size_D:
                D=GetRandomG(PLACES,2)
            randomg=D
            #print(randomg)
            J,HasPO,HasPL,PasPL=disj_divs (randomg,[l for l in range(1,NrRatpl+1)],None,NrRatpl,PLACES)
            #print(J,HasPO,HasPL,PasPL)
            w=str(randomg).replace("]","").replace("[","")
            print("intvec G="+w+";",file=AT)   
            #print("intvec G="+w+";")   
        elif u.find("intvec D")>=0:
            w=PallowedByH(HasPO)
            #print("w(allowed)=",w)
            wr=random.sample(w,size_D)
            wr.sort()
            wr
    #        w=str(randomD(1,9,6,iavoid)).replace("]","").replace("[","")
            w=str(wr).replace("]","").replace("[","")
            #print("w=",w)
            print("intvec D="+w+";",file=AT)
            #print("intvec D="+w+";")
        elif u.find("intvec m")>=0:
            intvecm=random.sample([i for i in range(1,20)],2)
            w=str(intvecm).replace("]","").replace("[","")
            print("intvec m="+w+";",file=AT)
        else:
            print(u,file=AT)
    AT.close()

    AT=open("AT_.txt","r")
    gene_text=AT.read()
    AT.close()
    G=Gene(gene_text)
    #print(G.gene_text)
    # コマンド実行
    result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
    if result.stdout.find("Vector basis successfully computed")>=0:
        print("OK")
        #print(result.stdout)
        G.get_result(result.stdout)
        G.distance=int(result.stdout.split("\n")[-2])
        print("distance=",result.stdout.split("\n")[-2],"current_max_distance=",current_max_distance)   # 標準出力
        if int(result.stdout.split("\n")[-2])>current_max_distance:
            current_max_distance=int(result.stdout.split("\n")[-2])
        OK=True
        return G
    else:
        OK=False
        print("FAILED")
        f=open("AT_.txt","r")
        print(f.readline())
        f.close()
        return G0
    #print(result.stdout)     
    #print(result.stderr)   # エラー出力
    #print(result.returncode)  # 終了コード

OK=True
import time
G=Gene(A0+C)
while OK==True:
    #RandomCSS_D_A()
    G=Mutate(G)
    time.sleep(5)

OK
distance=    1 current_max_distance= 0
OK
distance=    1 current_max_distance= 1
OK
distance=    1 current_max_distance= 1
OK
distance=    1 current_max_distance= 1


KeyboardInterrupt: 

In [ ]:
G.gene_text

In [ ]:
G=Gene(AS)
A=A0+C

In [ ]:
import subprocess
A=A0+C
A
AS=A.split("\n")
#print(AS)



def RandomCSS():    
    AT=open("AT_.txt","w")
    iavoid=None
    HasPO=None
    HasPL=None
    PasPL=None
    for u in AS:
    # WARNINGS: G  should satisfy @math{ 2*genus-2 < deg(G) < size(D) }, which is
    #           not checked by the algorithm.
        size_D=5
        if u.find("intvec G")>=0:
            D=GetRandomG(PLACES,2)
            while sum(D) >=size_D:
                D=GetRandomG(PLACES,2)
            print(D)
            randomg=D
            print(randomg,iavoid)
            J,HasPO,HasPL,PasPL=disj_divs (randomg,[l for l in range(1,NrRatpl+1)],None,NrRatpl,PLACES)
            print(J,HasPO,HasPL,PasPL)
            w=str(randomg).replace("]","").replace("[","")
            print("intvec G="+w+";",file=AT)   
            print("intvec G="+w+";")   
        elif u.find("intvec D")>=0:
            w=PallowedByH(HasPO)
            print("w(allowed)=",w)
            wr=random.sample(w,size_D)
            wr.sort()
            wr
    #        w=str(randomD(1,9,6,iavoid)).replace("]","").replace("[","")
            w=str(wr).replace("]","").replace("[","")
            print("w=",w)
            print("intvec D="+w+";",file=AT)
            print("intvec D="+w+";")
        else:
            print(u,file=AT)
    AT.close()

    AT=open("AT_.txt","r")
    gene_text=AT.read()
    AT.close()
    G=Gene(gene_text)
    
    # コマンド実行
    result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
    
    print(result.stdout)   # 標準出力
    if result.stdout.find("Vector basis successfully computed")>=0:
        print("OK")
        OK=True
    else:
        pass
        OK=False
        print("FAILED")
        f=open("AT_.txt","r")
        print(f.readline())
        f.close()
    
    #print(result.stderr)   # エラー出力
    print(result.returncode)  # 終了コード

OK=True
while OK==True:
    RandomCSS()

In [ ]:

import subprocess
A=A0+C
A
AS=A.split("\n")
#print(AS)
OK=True
while OK!=False:
    AT=open("AT_.txt","w")
    iavoid=None
    for u in AS:
        if u.find("intvec G")>=0:
            randomg,iavoid=getrandomG(1,4)
            print(randomg,iavoid)
            w=str(randomg).replace("]","").replace("[","")
            print("intvec G="+w+";",file=AT)   
            print("intvec G="+w+";",iavoid)   
        elif u.find("intvec D")>=0:
            w=str(randomD(1,9,6,iavoid)).replace("]","").replace("[","")
            #print(w)
            print("intvec D="+w+";",file=AT)
            print("intvec D="+w+";")
        else:
            print(u,file=AT)
    AT.close()

    
    # コマンド実行
    result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)
    
    print(result.stdout)   # 標準出力
    if result.stdout.find("Vector basis successfully computed")>=0:
        print("OK")
        OK=True
    else:
        pass
        OK=False
        print("FAILED")
        f=open("AT_.txt","r")
        print(f.readline())
        f.close()
    if result.stdout.find("wrong range")>=0:
        break
    #print(result.stderr)   # エラー出力
    print(result.returncode)  # 終了コード

In [ ]:
AS=A.split("\n")

In [ ]:
import subprocess

# コマンド実行
result = subprocess.run(["pwd"], capture_output=True, text=True)

print(result.stdout)   # 標準出力
print(result.stderr)   # エラー出力
print(result.returncode)  # 終了コード

In [ ]:

import subprocess

# コマンド実行
result = subprocess.run(["Singular -q AT_.txt"], shell=True,capture_output=True, text=True)

print(result.stdout)   # 標準出力
print(result.stderr)   # エラー出力
print(result.returncode)  # 終了コード


In [ ]:
import sympy
a=sympy.symbols("a")
Cfound=False
CM=[]
for u in result.stdout.split("\n"):
    if u.find('Vector basis successfully computed ')>=0:
        Cfound=True
    if Cfound==True and u.find(",")>=0:
        CM.append(eval("["+u+"]"))
#CM=sympy.Matrix(CM)
CM

In [ ]:
def min_wt_rmat (M):
    m=len(M)
    n=len(M[0])
    Hwt=0
    for j in range(n):
        if M[0][j]!=0:
            Hwt=Hwt+1
    minHwt=Hwt
    sizerows=[0]*m
    sizerows[0]=Hwt
    k=0
    for i in range(1,m):
        Hwt=0
        for j in range(n):
            if M[i][j]!=0:
                Hwt=Hwt+1
        sizerows[i]=Hwt
        if Hwt<minHwt:
            minHwt=Hwt
            k=i
    
    return k,minHwt,sizerows

def MySubmat(intvec, M):
    A=[]
    for u in intvec:
        A.append(M[u])
    return A
min_wt_rmat(CM)


In [ ]:
import random
RR=random.sample(range(len(CM)),2)
RR.sort()
RR

In [ ]:
min_wt_rmat(MySubmat(RR,CM))

In [57]:
import random
from typing import List, Callable, Tuple

# 個体（遺伝子） = 整数のリスト
Individual = List[int]


class GeneticAlgorithm:
    def __init__(
        self,
        gene_length: int,
        gene_min: int,
        gene_max: int,
        population_size: int,
        fitness_func: Callable[[Individual], float],
        crossover_rate: float = 0.8,
        mutation_rate: float = 0.1,
        mutation_strength: int = 1,
    ):
        self.gene_length = gene_length
        self.gene_min = gene_min
        self.gene_max = gene_max
        self.population_size = population_size
        self.fitness_func = fitness_func
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.mutation_strength = mutation_strength

        self.population = self._initialize_population()

    # 初期集団生成
    def _initialize_population(self) -> List:
        return [
            Gene(A0+C)
            for _ in range(self.population_size)
        ]

    # 評価
    def _evaluate(self) -> List[Tuple[Individual, float]]:
        return [(ind, self.fitness_func(ind)) for ind in self.population]

    # トーナメント選択
    def _selection(self) -> Individual:
        k = 3
        selected = random.sample(self.population, k)
        selected.sort(key=self.fitness_func, reverse=True)
        return selected[0]

    # 交叉（1点交叉）
    def _crossover(self, p1: Individual, p2: Individual) -> Individual:
        if random.random() > self.crossover_rate:
            return p1[:]

        point = random.randint(1, self.gene_length - 1)
        return p1[:point] + p2[point:]

    # 突然変異
    def _mutation(self, ind: Individual) -> Individual:
        new_ind = ind[:]
        for i in range(self.gene_length):
            if random.random() < self.mutation_rate:
                new_ind[i] += random.randint(
                    -self.mutation_strength, self.mutation_strength)
                # 範囲制限
                new_ind[i] = max(self.gene_min, min(self.gene_max, new_ind[i]))
        return new_ind

    # 1世代進化
    def step(self):
        new_population = []

        # エリート保存（最良個体1つ）
        evaluated = self._evaluate()
        best = max(evaluated, key=lambda x: x[1])[0]
        new_population.append(best)

        # 残り生成
        while len(new_population) < self.population_size:
            p1 = self._selection()
            p2 = self._selection()

            child = self._crossover(p1, p2)
            child = self._mutation(child)

            new_population.append(child)

        self.population = new_population

    # 実行
    def run(self, generations: int, verbose: bool = True):
        for gen in range(generations):
            self.step()
            best, fitness = self.get_best()

            if verbose:
                print(f"Gen {gen}: Best Fitness = {fitness}")

        return self.get_best()

    # 最良個体取得
    def get_best(self) -> Tuple[Individual, float]:
        evaluated = self._evaluate()
        return max(evaluated, key=lambda x: x[1])

def fitness(individual):
    return individual.distance


ga = GeneticAlgorithm(
    gene_length=10,
    gene_min=0,
    gene_max=10,
    population_size=50,
    fitness_func=fitness
)

In [43]:
gm.distance

1

In [53]:
gm.gene_text

'LIB "mybrnoeth.lib";\nLIB "myprocs.txt";\nring s=2,(x,y),lp;\nlist HC=Adj_div(x3+y2+y);\nHC=NSplaces(1..2,HC);\nHC=extcurve(2,HC);\ndef ER=HC[1][4];\nsetring ER;\nintvec G=0, 0, 0, 2, 1, 0;\nintvec D=1, 2, 3, 8, 9;\nmatrix C=AGcode_L(G,D,HC);\nprint(C);\nprint("Hamming_wt");\nprint(min_wt_rmat(C));\nprint("C");\nprint(C);\nprint("check_fully_one_rows");\nintvec a=check_fully_one_rows(C);\nprint(a);\nif (nrows(C)>1){C=MySubmat(a,C);\nprint("C [The row (1,1,...,1) is removed]");\nprint(C);\n}print("Check_fully_zero_columns");\nprint(check_fully_zero_columns(C));\nintvec b=check_fully_zero_columns(C);\nif (1){C=MySubmat_cols(b, C);\nprint("C [Fully zero colums are removed.]");\nprint(C);\n}print(my_min_wt_rmat(C));\nprint("SC");\nintvec m=6, 12;\nmatrix SC=MySubmat(m,C);\nprint(SC);\nprint("Ker(C)");\nprint(MyKer(C));\nprint("Ker(SC)");\nprint(MyKer(SC));\nmatrix KC=MyKer(C);\nmatrix KSC=MyKer(SC);\nprint("KSC\\KC");\nprint(MySupplement(KC,KSC));\nprint(my_min_wt_rmat(MySupplement(KC,KSC

In [54]:
ga.population[4].gene_text

'LIB "mybrnoeth.lib";\nLIB "myprocs.txt";\nring s=2,(x,y),lp;\nlist HC=Adj_div(x3+y2+y);\nHC=NSplaces(1..2,HC);\nHC=extcurve(2,HC);\ndef ER=HC[1][4];\nsetring ER;\nintvec G=5;\nintvec D=2,3,4,5,6,7,8,9; \nmatrix C=AGcode_L(G,D,HC);\nprint(C);\nprint("Hamming_wt");\nprint(min_wt_rmat(C));\nprint("C");\nprint(C);\nprint("check_fully_one_rows");\nintvec a=check_fully_one_rows(C);\nprint(a);\nif (nrows(C)>1){C=MySubmat(a,C);\nprint("C [The row (1,1,...,1) is removed]");\nprint(C);\n}print("Check_fully_zero_columns");\nprint(check_fully_zero_columns(C));\nintvec b=check_fully_zero_columns(C);\nif (1){C=MySubmat_cols(b, C);\nprint("C [Fully zero colums are removed.]");\nprint(C);\n}print(my_min_wt_rmat(C));\nprint("SC");\nintvec m=1,2;\nmatrix SC=MySubmat(m,C);\nprint(SC);\nprint("Ker(C)");\nprint(MyKer(C));\nprint("Ker(SC)");\nprint(MyKer(SC));\nmatrix KC=MyKer(C);\nmatrix KSC=MyKer(SC);\nprint("KSC\\KC");\nprint(MySupplement(KC,KSC));\nprint(my_min_wt_rmat(MySupplement(KC,KSC)));\nprint("C

In [ ]:
ga.population[0]

In [64]:
[(ind, fitness(ind)) for ind in ga.population]

[(<__main__.Gene at 0x74137442bda0>, 2),
 (<__main__.Gene at 0x741376439bb0>, 1),
 (<__main__.Gene at 0x741374b554c0>, 1),
 (<__main__.Gene at 0x741374b549e0>, 1),
 (<__main__.Gene at 0x741374543230>, 1),
 (<__main__.Gene at 0x741374540c20>, 0),
 (<__main__.Gene at 0x741374b57320>, 1),
 (<__main__.Gene at 0x741374b57500>, 1),
 (<__main__.Gene at 0x7413747fc6b0>, 1),
 (<__main__.Gene at 0x741374b4eb10>, 1),
 (<__main__.Gene at 0x741374b57260>, 1),
 (<__main__.Gene at 0x741374540650>, 1),
 (<__main__.Gene at 0x74137442b440>, 1),
 (<__main__.Gene at 0x74137648c620>, 1),
 (<__main__.Gene at 0x741374542390>, 2),
 (<__main__.Gene at 0x741374543a40>, 1),
 (<__main__.Gene at 0x7413745405c0>, 1),
 (<__main__.Gene at 0x741374b56030>, 1),
 (<__main__.Gene at 0x7413747fd850>, 1),
 (<__main__.Gene at 0x7413747fc2f0>, 1),
 (<__main__.Gene at 0x7413747fcc80>, 1),
 (<__main__.Gene at 0x7413747fc470>, 1),
 (<__main__.Gene at 0x7413747fe060>, 1),
 (<__main__.Gene at 0x7413747fc980>, 1),
 (<__main__.Gene

In [ ]:
for gm in A:
    gm=Mutate(gm)
    print(gm.distance)

In [63]:

for i,gm in enumerate(ga.population):
    ga.population[i]=Mutate(gm)
    print(ga.population[i].distance)

OK
distance=    2 current_max_distance= 3
2
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    0 current_max_distance= 3
0
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    2 current_max_distance= 3
2
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_distance= 3
1
OK
distance=    1 current_max_di

In [46]:
gm.distance

1

In [1]:
best_individual, best_score = ga.run(generations=100)

print("Best:", best_individual)
print("Score:", best_score)

Gen 0: Best Fitness = 76
Gen 1: Best Fitness = 78
Gen 2: Best Fitness = 83
Gen 3: Best Fitness = 86
Gen 4: Best Fitness = 87
Gen 5: Best Fitness = 91
Gen 6: Best Fitness = 91
Gen 7: Best Fitness = 92
Gen 8: Best Fitness = 93
Gen 9: Best Fitness = 93
Gen 10: Best Fitness = 96
Gen 11: Best Fitness = 96
Gen 12: Best Fitness = 98
Gen 13: Best Fitness = 98
Gen 14: Best Fitness = 99
Gen 15: Best Fitness = 99
Gen 16: Best Fitness = 100
Gen 17: Best Fitness = 100
Gen 18: Best Fitness = 100
Gen 19: Best Fitness = 100
Gen 20: Best Fitness = 100
Gen 21: Best Fitness = 100
Gen 22: Best Fitness = 100
Gen 23: Best Fitness = 100
Gen 24: Best Fitness = 100
Gen 25: Best Fitness = 100
Gen 26: Best Fitness = 100
Gen 27: Best Fitness = 100
Gen 28: Best Fitness = 100
Gen 29: Best Fitness = 100
Gen 30: Best Fitness = 100
Gen 31: Best Fitness = 100
Gen 32: Best Fitness = 100
Gen 33: Best Fitness = 100
Gen 34: Best Fitness = 100
Gen 35: Best Fitness = 100
Gen 36: Best Fitness = 100
Gen 37: Best Fitness = 100
